# Week 2 - Supervised Machine Learning Models

## Objective
This notebook demonstrates supervised learning using regression and classification models. It includes data loading, preprocessing, model training, evaluation, visualization, and comparison.

## Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_preprocessing import clean_dataset, load_dataset, prepare_regression_data, prepare_classification_data, split_dataset
from src.regression_models import train_linear_regression, train_decision_tree_regressor, train_random_forest_regressor
from src.classification_models import train_logistic_regression, train_decision_tree_classifier, train_random_forest_classifier, train_knn_classifier
from src.evaluation import calculate_regression_metrics, calculate_classification_metrics, save_results_dataframe

## Load Dataset

In [ ]:
project_root = Path.cwd()
dataset_path = project_root / 'data' / 'dataset.csv'
df = clean_dataset(load_dataset(dataset_path))
print('Shape:', df.shape)
print(df.head())

## Understand the Dataset

In [ ]:
print(df.info())
print(df.describe().round(3).to_string())
print('Missing values:\n', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())

## Data Cleaning

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
print('After cleaning:', df.shape)

## Exploratory Data Analysis

In [ ]:
sns.histplot(df['quality'], bins=10, kde=True)
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.show()

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', linewidths=.5)
plt.title('Correlation Heatmap')
plt.show()

## Data Preprocessing

In [ ]:
X_reg, y_reg = prepare_regression_data(df)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = split_dataset(X_reg, y_reg, task='regression')

X_cls, y_cls = prepare_classification_data(df, threshold=6)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = split_dataset(X_cls, y_cls, task='classification')
print('Regression train/test sizes:', X_train_reg.shape[0], X_test_reg.shape[0])
print('Classification train/test sizes:', X_train_cls.shape[0], X_test_cls.shape[0])

## Regression - Linear Regression

In [ ]:
linear_model = train_linear_regression(X_train_reg, y_train_reg)
linear_pred = linear_model.predict(X_test_reg)
linear_metrics = calculate_regression_metrics(y_test_reg, linear_pred)
print(linear_metrics)

## Regression - Decision Tree

In [ ]:
tree_model = train_decision_tree_regressor(X_train_reg, y_train_reg, random_state=42)
tree_pred = tree_model.predict(X_test_reg)
tree_metrics = calculate_regression_metrics(y_test_reg, tree_pred)
print(tree_metrics)

## Regression - Random Forest

In [ ]:
rf_model = train_random_forest_regressor(X_train_reg, y_train_reg, random_state=42, n_estimators=200)
rf_pred = rf_model.predict(X_test_reg)
rf_metrics = calculate_regression_metrics(y_test_reg, rf_pred)
print(rf_metrics)

## Regression Model Evaluation

In [ ]:
regression_results = [
    {'Model': 'Linear Regression', **linear_metrics},
    {'Model': 'Decision Tree Regressor', **tree_metrics},
    {'Model': 'Random Forest Regressor', **rf_metrics},
]
regression_df = pd.DataFrame(regression_results)
print(regression_df)

## Classification Target Creation

In [ ]:
classification_df = df.copy()
classification_df['quality_label'] = (classification_df['quality'] >= 6).astype(int)
print(classification_df['quality_label'].value_counts())

## Classification - Logistic Regression

In [ ]:
logistic_model = train_logistic_regression(X_train_cls, y_train_cls, random_state=42)
logistic_pred = logistic_model.predict(X_test_cls)
logistic_metrics = calculate_classification_metrics(y_test_cls, logistic_pred)
print(logistic_metrics)

## Classification - Decision Tree

In [ ]:
dt_model = train_decision_tree_classifier(X_train_cls, y_train_cls, random_state=42)
dt_pred = dt_model.predict(X_test_cls)
dt_metrics = calculate_classification_metrics(y_test_cls, dt_pred)
print(dt_metrics)

## Classification - Random Forest

In [ ]:
rf_cls_model = train_random_forest_classifier(X_train_cls, y_train_cls, random_state=42, n_estimators=200)
rf_cls_pred = rf_cls_model.predict(X_test_cls)
rf_cls_metrics = calculate_classification_metrics(y_test_cls, rf_cls_pred)
print(rf_cls_metrics)

## Classification - KNN

In [ ]:
knn_model = train_knn_classifier(X_train_cls, y_train_cls, random_state=42, n_neighbors=5)
knn_pred = knn_model.predict(X_test_cls)
knn_metrics = calculate_classification_metrics(y_test_cls, knn_pred)
print(knn_metrics)

## Classification Model Evaluation

In [ ]:
classification_results = [
    {'Model': 'Logistic Regression', **logistic_metrics},
    {'Model': 'Decision Tree Classifier', **dt_metrics},
    {'Model': 'Random Forest Classifier', **rf_cls_metrics},
    {'Model': 'KNN Classifier', **knn_metrics},
]
classification_df = pd.DataFrame(classification_results)
print(classification_df)

## Model Comparison

In [ ]:
combined = pd.concat([
    pd.DataFrame(regression_results),
    pd.DataFrame(classification_results),
], ignore_index=True)
print(combined)

## Results and Observations

In [ ]:
print('Regression best model by R2:', regression_df.loc[regression_df['R2'].idxmax(), 'Model'])
print('Classification best model by F1:', classification_df.loc[classification_df['F1'].idxmax(), 'Model'])

## Results Summary

### Regression
| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 0.504141 | 0.656513 | 0.391536 |
| Decision Tree Regressor | 0.650735 | 0.945189 | -0.261206 |
| Random Forest Regressor | 0.467445 | 0.617338 | 0.461984 |

### Classification
| Model | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Logistic Regression | 0.735294 | 0.760870 | 0.729167 | 0.744681 |
| Decision Tree Classifier | 0.625000 | 0.641892 | 0.659722 | 0.650685 |
| Random Forest Classifier | 0.757353 | 0.786765 | 0.743056 | 0.764286 |
| KNN Classifier | 0.687500 | 0.690323 | 0.743056 | 0.715719 |

The Random Forest Regressor had the lowest RMSE and highest R², while the Random Forest Classifier achieved the best accuracy and F1 score. Different metrics emphasize different aspects of performance, so a model should be judged using the metric that matches the problem objective.

## Conclusion
This notebook shows how supervised learning can be applied to real-world data. It uses regression to predict a continuous target and classification to separate wines into higher or lower quality groups. The project compares multiple models and interprets the results using appropriate performance metrics.